[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefHosni/prompt-engineering-book-2e/blob/main/notebooks/ch07_task_playbook.ipynb)

# Chapter 7 — The Task Playbook

One input, six tasks. The chapter's argument is that these are not six
techniques; they are one technique with the risk moved around.

On the left of that axis the failure is **omission** — something is dropped
silently and the output still looks complete. On the right it is
**invention**. The middle two should preserve meaning exactly, which makes
them easiest to check and easiest to get quietly wrong.

In [1]:
# Colab only: fetch the shared helpers and install the SDKs.
# Running locally? Skip this cell -- you already have them.
import os, sys
if "google.colab" in sys.modules:
    !pip -q install anthropic openai python-dotenv
    !git clone -q https://github.com/youssefHosni/prompt-engineering-book-2e /content/peb-book
    sys.path.insert(0, "/content/peb-book")
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")


In [2]:
from peb import ask
from peb.fixtures import REVIEW

print(REVIEW)

Ordered the Nordic 400 on the 3rd, arrived on the 11th which is longer than the
3 days advertised. Machine itself is excellent - the burr grinder is much quieter
than my old one and the espresso is genuinely good. Two problems though. The drip
tray warped after about a week, it's very thin plastic. And the manual is clearly
a bad translation, there's a step about "descaling the vapour wand" that makes no
sense. Support answered in about 4 hours which was better than I expected. Would
still recommend it but be prepared to buy a replacement tray. Order ref KL-449021.


In [3]:
def run(label, template, **checks):
    text, _ = ask(template.format(review=REVIEW), max_tokens=4000)
    print(f"--- {label}")
    print(text.strip()[:520])
    for name, fn in checks.items():
        print(f"  [check] {name}: {fn(text)}")
    print()

## §7.1 Summarising: the purpose is the prompt

There is no such thing as *the* summary of a text — only the summary for
somebody, for something. Two readers, same review.

In [4]:
pm = """Summarise this review in 2 sentences for a product manager deciding
what to fix next. Report only faults and their severity. Ignore praise and
logistics.

Review: ```{review}```"""

ops = """Summarise this review in 2 sentences for a logistics lead.
Report only delivery and support performance. Ignore the product itself.

Review: ```{review}```"""

run("summarise (product manager)", pm)
run("summarise (logistics lead)", ops)

--- summarise (product manager)
Drip tray warped within a week due to excessively thin plastic — high severity, as it's a physical defect that forces customers to buy a replacement part. The manual contains a mistranslation, including an incomprehensible step on "descaling the vapour wand" — lower severity, but it blocks correct maintenance guidance.



--- summarise (logistics lead)
**Order KL-449021:** Delivery missed the advertised 3-day window by a wide margin — ordered on the 3rd, delivered on the 11th (8 days). Support performance was strong, with a response in roughly 4 hours, exceeding the customer's expectations; the only related gripe was poorly translated documentation, which may drive avoidable support contacts.



Those two summaries share no content. Neither is wrong; they answer different
questions, and two clauses made the difference.

## §7.2 Extraction — the keys are fixed, so the check is trivial

In [5]:
extract = """Extract from the review. Use exactly these keys, one per line,
in this order. Use the literal string NONE if a value is absent.

order_ref | ordered_date | arrived_date | support_hours

Review: ```{review}```"""

run("extract", extract,
    all_keys=lambda t: all(k in t for k in
                           ("order_ref", "ordered_date", "arrived_date", "support_hours")))

--- extract
order_ref | KL-449021
ordered_date | the 3rd
arrived_date | the 11th
support_hours | 4
  [check] all_keys: True



## §7.3 Classification

In [6]:
classify = """Classify this review. Reply with exactly these three lines
and nothing else:

sentiment: positive | negative | mixed
recommends: yes | no
faults: a comma-separated list, or NONE

Review: ```{review}```"""

run("classify", classify,
    three_lines=lambda t: len([l for l in t.strip().splitlines() if l.strip()]) == 3)

--- classify
sentiment: mixed
recommends: yes
faults: slow delivery (8 days vs 3 advertised), drip tray warped after a week, drip tray plastic too thin, poorly translated/confusing manual
  [check] three_lines: True



## §7.4 Transformation — the risk is silent loss

`must_keep` is the whole technique. Decide what must survive, then check it in
code rather than reading the output and feeling reassured.

In [7]:
must_keep = ["KL-449021", "drip tray", "manual", "support"]

transform = """Rewrite the review as an internal support ticket. Neutral register,
no first person, keep every order reference and part name exactly as written.

Review: ```{review}```"""

run("transform", transform,
    kept=lambda t: [k for k in must_keep if k.lower() not in t.lower()] or "all kept")

--- transform
**SUPPORT TICKET**

**Order Reference:** KL-449021
**Product:** Nordic 400
**Ticket Type:** Post-delivery feedback — multiple issues

---

**1. Delivery delay**
Order placed on the 3rd; delivery completed on the 11th (8 days). Advertised delivery window is 3 days. Discrepancy of approximately 5 days beyond the stated timeframe.

**2. Hardware defect — drip tray**
The drip tray warped after approximately one week of use. Customer attributes this to thin plastic construction. Replacement part likely required; custome
  [check] kept: all kept



## §7.5 Translation — the same check, across a script boundary

In [8]:
translate = """Translate the review into Modern Standard Arabic. Keep product
names, order references and part numbers in Latin script, unchanged.

Review: ```{review}```"""

run("translate", translate,
    latin_kept=lambda t: "KL-449021" in t and "Nordic 400" in t)

--- translate
المراجعة:

```
طلبت الـ Nordic 400 في الثالث من الشهر، ووصل في الحادي عشر، أي مدة أطول من الأيام الثلاثة المُعلن عنها. الآلة نفسها ممتازة؛ فمطحنة الأقراص أهدأ بكثير من مطحنتي القديمة، والإسبريسو لذيذ فعلًا. لكن هناك مشكلتان. صينية التنقيط تشوّهت بعد أسبوع تقريبًا، فهي مصنوعة من بلاستيك رقيق جدًا. كما أن دليل الاستخدام ترجمة سيئة بشكل واضح، إذ يحتوي على خطوة عن "إزالة الترسبات الكلسية من عصا البخار" لا معنى لها. أجاب فريق الدعم في غضون أربع ساعات تقريبًا، وهو أفضل مما توقعت. ما زلت أنصح بها، لكن استعدّ لشراء صينية ب
  [check] latin_kept: True



## §7.6 Expansion — the highest-risk task, so the most constraints

This is the far right of the axis. Every constraint here exists because its
absence produced something wrong once.

In [9]:
expand = """Write a reply to this customer.
Length: 60 to 80 words.
Acknowledge the late delivery and the drip tray.
Offer a replacement tray. Do not offer a refund.
Do not promise a delivery date.
Sign as 'Kestrel Support'.

Review: ```{review}```"""

run("expand", expand,
    length=lambda t: len(t.split()),
    no_refund=lambda t: "refund" not in t.lower(),
    signed=lambda t: "Kestrel Support" in t)

--- expand
Thank you for the detailed feedback on order KL-449021. We're sorry the Nordic 400 took longer than the advertised three days to reach you, and that the drip tray warped so soon. We'd like to send you a replacement tray at no cost — just confirm your address and we'll arrange it. We've also flagged the manual's descaling section for review. Glad to hear the grinder and espresso are living up to expectations.

Kestrel Support
  [check] length: 75
  [check] no_refund: True
  [check] signed: True

